# Task description
- Classify the speakers of given features.
- Main goal: Learn how to use transformer.
- Baselines:
  - Easy: Run sample code and know how to use transformer.
  - Medium: Know how to adjust parameters of transformer.
  - Hard: Construct [conformer](https://arxiv.org/abs/2005.08100) which is a variety of transformer.

- Other links
  - Kaggle: [link](https://www.kaggle.com/t/859c9ca9ede14fdea841be627c412322)
  - Slide: [link](https://speech.ee.ntu.edu.tw/~hylee/ml/ml2021-course-data/hw/HW04/HW04.pdf)
  - Data: [link](https://drive.google.com/file/d/1T0RPnu-Sg5eIPwQPfYysipfcz81MnsYe/view?usp=sharing)
  - Video (Chinese): [link](https://www.youtube.com/watch?v=EPerg2UnGaI)
  - Video (English): [link](https://www.youtube.com/watch?v=Gpz6AUvCak0)
  - Solution for downloading dataset fail.: [link](https://drive.google.com/drive/folders/13T0Pa_WGgQxNkqZk781qhc5T9-zfh19e?usp=sharing)

# Download dataset
- Please follow [here](https://drive.google.com/drive/folders/13T0Pa_WGgQxNkqZk781qhc5T9-zfh19e?usp=sharing) to download data
- Data is [here](https://drive.google.com/file/d/1gaFy8RaQVUEXo2n0peCBR5gYKCB-mNHc/view?usp=sharing)

In [20]:
# 1. 在 Colab 运行这段代码挂载云盘
from google.colab import drive
drive.mount('/content/drive')

# 2. 挂载后，直接解压（速度极快，且不会有权限报错）
!unzip /content/drive/MyDrive/Dataset.zip -d /content/dataset

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Archive:  /content/drive/MyDrive/Dataset.zip
replace /content/dataset/Dataset/uttr-f2e2dc9d3a6c471abd6464a316d1e21a.pt? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

# Data

## Dataset
- Original dataset is [Voxceleb1](https://www.robots.ox.ac.uk/~vgg/data/voxceleb/).
- The [license](https://creativecommons.org/licenses/by/4.0/) and [complete version](https://www.robots.ox.ac.uk/~vgg/data/voxceleb/files/license.txt) of Voxceleb1.
- We randomly select 600 speakers from Voxceleb1.
- Then preprocess the raw waveforms into mel-spectrograms.

- Args:
  - data_dir: The path to the data directory.
  - metadata_path: The path to the metadata.
  - segment_len: The length of audio segment for training.
- The architecture of data directory \\
  - data directory \\
  |---- metadata.json \\
  |---- testdata.json \\
  |---- mapping.json \\
  |---- uttr-{random string}.pt \\

- The information in metadata
  - "n_mels": The dimention of mel-spectrogram.
  - "speakers": A dictionary.
    - Key: speaker ids.
    - value: "feature_path" and "mel_len"


For efficiency, we segment the mel-spectrograms into segments in the traing step.

In [2]:
import os
import json
import torch
import random
from pathlib import Path
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence


class myDataset(Dataset):
  def __init__(self, data_dir, segment_len=128):
    self.data_dir = data_dir
    self.segment_len = segment_len

    # Load the mapping from speaker neme to their corresponding id.
    mapping_path = Path(data_dir) / "mapping.json"
    mapping = json.load(mapping_path.open())
    self.speaker2id = mapping["speaker2id"]

    # Load metadata of training data.
    metadata_path = Path(data_dir) / "metadata.json"
    metadata = json.load(open(metadata_path))["speakers"]

    # Get the total number of speaker.
    self.speaker_num = len(metadata.keys())
    self.data = []
    for speaker in metadata.keys():
      for utterances in metadata[speaker]:
        self.data.append([utterances["feature_path"], self.speaker2id[speaker]])

  def __len__(self):
    return len(self.data)

  def __getitem__(self, index):
    feat_path, speaker = self.data[index]
    # Load preprocessed mel-spectrogram.
    mel = torch.load(os.path.join(self.data_dir, feat_path))

    # Segmemt mel-spectrogram into "segment_len" frames.
    if len(mel) > self.segment_len:
      # Randomly get the starting point of the segment.
      start = random.randint(0, len(mel) - self.segment_len)
      # Get a segment with "segment_len" frames.
      mel = torch.FloatTensor(mel[start:start+self.segment_len])
    else:
      mel = torch.FloatTensor(mel)
    # Turn the speaker id into long for computing loss later.
    speaker = torch.FloatTensor([speaker]).long()
    return mel, speaker

  def get_speaker_number(self):
    return self.speaker_num

## Dataloader
- Split dataset into training dataset(90%) and validation dataset(10%).
- Create dataloader to iterate the data.


In [3]:
import torch
from torch.utils.data import DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence


def collate_batch(batch):
  # Process features within a batch.
  """Collate a batch of data."""
  mel, speaker = zip(*batch)
  # Because we train the model batch by batch, we need to pad the features in the same batch to make their lengths the same.
  mel = pad_sequence(mel, batch_first=True, padding_value=-20)    # pad log 10^(-20) which is very small value.
  # mel: (batch size, length, 40)
  return mel, torch.FloatTensor(speaker).long()


def get_dataloader(data_dir, batch_size, n_workers):
  """Generate dataloader"""
  dataset = myDataset(data_dir)
  speaker_num = dataset.get_speaker_number()
  # Split dataset into training dataset and validation dataset
  trainlen = int(0.9 * len(dataset))
  lengths = [trainlen, len(dataset) - trainlen]
  trainset, validset = random_split(dataset, lengths)

  train_loader = DataLoader(
    trainset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
    num_workers=n_workers,
    pin_memory=True,
    collate_fn=collate_batch,
  )
  valid_loader = DataLoader(
    validset,
    batch_size=batch_size,
    num_workers=n_workers,
    drop_last=True,
    pin_memory=True,
    collate_fn=collate_batch,
  )

  return train_loader, valid_loader, speaker_num


# Model
- TransformerEncoderLayer:
  - Base transformer encoder layer in [Attention Is All You Need](https://arxiv.org/abs/1706.03762)
  - Parameters:
    - d_model: the number of expected features of the input (required).

    - nhead: the number of heads of the multiheadattention models (required).

    - dim_feedforward: the dimension of the feedforward network model (default=2048).

    - dropout: the dropout value (default=0.1).

    - activation: the activation function of intermediate layer, relu or gelu (default=relu).

- TransformerEncoder:
  - TransformerEncoder is a stack of N transformer encoder layers
  - Parameters:
    - encoder_layer: an instance of the TransformerEncoderLayer() class (required).

    - num_layers: the number of sub-encoder-layers in the encoder (required).

    - norm: the layer normalization component (optional).

In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ConformerBlock(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=256, kernel_size=31, dropout=0.1):
        super().__init__()
        # 1. Feed Forward (Half-step)
        self.ffn1 = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, dim_feedforward),
            nn.SiLU(), # Swish
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
            nn.Dropout(dropout)
        )

        # 2. Self-Attention
        self.attn_norm = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)

        # 3. Convolution Module
        self.conv_norm = nn.LayerNorm(d_model)
        self.conv = nn.Sequential(
            nn.Conv1d(d_model, 2 * d_model, kernel_size=1),
            nn.GLU(dim=1),
            nn.Conv1d(d_model, d_model, kernel_size=kernel_size,
                      padding=(kernel_size - 1) // 2, groups=d_model),
            nn.BatchNorm1d(d_model),
            nn.SiLU(),
            nn.Conv1d(d_model, d_model, kernel_size=1),
            nn.Dropout(dropout)
        )

        # 4. Feed Forward (Half-step)
        self.ffn2 = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, dim_feedforward),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(dim_feedforward, d_model),
            nn.Dropout(dropout)
        )
        self.final_norm = nn.LayerNorm(d_model)

    def forward(self, x):
        x = x + 0.5 * self.ffn1(x)

        residual = x
        x = self.attn_norm(x)
        x, _ = self.attn(x, x, x)
        x = residual + x

        residual = x
        x = self.conv_norm(x)
        x = x.transpose(1, 2)
        x = self.conv(x)
        x = x.transpose(1, 2)
        x = residual + x

        x = x + 0.5 * self.ffn2(x)
        return self.final_norm(x)

class Classifier(nn.Module):
    def __init__(self, d_model=256, n_spks=600, dropout=0.1, num_layers=4):
        super().__init__()
        self.prenet = nn.Linear(40, d_model)

        # --- 改进 1：堆叠多个 Conformer Block ---
        # 使用 nn.ModuleList 管理多层网络
        self.encoder = nn.ModuleList([
            ConformerBlock(
                d_model=d_model,
                nhead=4,
                dim_feedforward=d_model * 2, # 通常设为 d_model 的 2-4 倍
                dropout=dropout
            ) for _ in range(num_layers)
        ])

        # --- 改进 2：预测层优化 ---
        # 增加一个线性层或使用更好的结构来拟合 Speaker Embedding
        self.pred_layer = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.BatchNorm1d(d_model),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, n_spks),
        )

    def forward(self, mels):
        """
        mels: (batch size, length, 40)
        """
        out = self.prenet(mels)

        # --- 循环执行每一层 Conformer ---
        for layer in self.encoder:
            out = layer(out)

        # --- 改进 3：统计池化 (Statistical Pooling) ---
        # 声纹识别中，均值 + 标准差 通常比单纯的均值效果更好
        # 这里为了简单维持 mean pooling，但你可以尝试 torch.cat([out.mean(1), out.std(1)], dim=1)
        stats = out.mean(dim=1)

        out = self.pred_layer(stats)
        return out


# Learning rate schedule
- For transformer architecture, the design of learning rate schedule is different from that of CNN.
- Previous works show that the warmup of learning rate is useful for training models with transformer architectures.
- The warmup schedule
  - Set learning rate to 0 in the beginning.
  - The learning rate increases linearly from 0 to initial learning rate during warmup period.

In [22]:
import math

import torch
from torch.optim import Optimizer
from torch.optim.lr_scheduler import LambdaLR


def get_cosine_schedule_with_warmup(
  optimizer: Optimizer,
  num_warmup_steps: int,
  num_training_steps: int,
  num_cycles: float = 0.5,
  last_epoch: int = -1,
):
  """
  Create a schedule with a learning rate that decreases following the values of the cosine function between the
  initial lr set in the optimizer to 0, after a warmup period during which it increases linearly between 0 and the
  initial lr set in the optimizer.

  Args:
    optimizer (:class:`~torch.optim.Optimizer`):
      The optimizer for which to schedule the learning rate.
    num_warmup_steps (:obj:`int`):
      The number of steps for the warmup phase.
    num_training_steps (:obj:`int`):
      The total number of training steps.
    num_cycles (:obj:`float`, `optional`, defaults to 0.5):
      The number of waves in the cosine schedule (the defaults is to just decrease from the max value to 0
      following a half-cosine).
    last_epoch (:obj:`int`, `optional`, defaults to -1):
      The index of the last epoch when resuming training.

  Return:
    :obj:`torch.optim.lr_scheduler.LambdaLR` with the appropriate schedule.
  """

  def lr_lambda(current_step):
    # Warmup
    if current_step < num_warmup_steps:
      return float(current_step) / float(max(1, num_warmup_steps))
    # decadence
    progress = float(current_step - num_warmup_steps) / float(
      max(1, num_training_steps - num_warmup_steps)
    )
    return max(
      0.0, 0.5 * (1.0 + math.cos(math.pi * float(num_cycles) * 2.0 * progress))
    )

  return LambdaLR(optimizer, lr_lambda, last_epoch)


# Model Function
- Model forward function.

In [23]:
import torch


def model_fn(batch, model, criterion, device):
  """Forward a batch through the model."""

  mels, labels = batch
  mels = mels.to(device)
  labels = labels.to(device)

  outs = model(mels)

  loss = criterion(outs, labels)

  # Get the speaker id with highest probability.
  preds = outs.argmax(1)
  # Compute accuracy.
  accuracy = torch.mean((preds == labels).float())

  return loss, accuracy


# Validate
- Calculate accuracy of the validation set.

In [24]:
from tqdm import tqdm
import torch


def valid(dataloader, model, criterion, device):
  """Validate on validation set."""

  model.eval()
  running_loss = 0.0
  running_accuracy = 0.0
  pbar = tqdm(total=len(dataloader.dataset), ncols=0, desc="Valid", unit=" uttr")

  for i, batch in enumerate(dataloader):
    with torch.no_grad():
      loss, accuracy = model_fn(batch, model, criterion, device)
      running_loss += loss.item()
      running_accuracy += accuracy.item()

    pbar.update(dataloader.batch_size)
    pbar.set_postfix(
      loss=f"{running_loss / (i+1):.2f}",
      accuracy=f"{running_accuracy / (i+1):.2f}",
    )

  pbar.close()
  model.train()

  return running_accuracy / len(dataloader)


# Main function

In [25]:
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, random_split


def parse_args():
  """arguments"""
  config = {
    "data_dir": "/content/dataset/Dataset",
    "save_path": "model.ckpt",
    "batch_size": 32,
    "n_workers": 8,
    "valid_steps": 2000,
    "warmup_steps": 1000,
    "save_steps": 10000,
    "total_steps": 70000,
  }

  return config


def main(
  data_dir,
  save_path,
  batch_size,
  n_workers,
  valid_steps,
  warmup_steps,
  total_steps,
  save_steps,
):
  """Main function."""
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  print(f"[Info]: Use {device} now!")

  train_loader, valid_loader, speaker_num = get_dataloader(data_dir, batch_size, n_workers)
  train_iterator = iter(train_loader)
  print(f"[Info]: Finish loading data!",flush = True)

  model = Classifier(n_spks=speaker_num).to(device)
  criterion = nn.CrossEntropyLoss()
  optimizer = AdamW(model.parameters(), lr=1e-3)
  scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
  print(f"[Info]: Finish creating model!",flush = True)

  best_accuracy = -1.0
  best_state_dict = None

  pbar = tqdm(total=valid_steps, ncols=0, desc="Train", unit=" step")

  for step in range(total_steps):
    # Get data
    try:
      batch = next(train_iterator)
    except StopIteration:
      train_iterator = iter(train_loader)
      batch = next(train_iterator)

    loss, accuracy = model_fn(batch, model, criterion, device)
    batch_loss = loss.item()
    batch_accuracy = accuracy.item()

    # Updata model
    loss.backward()
    optimizer.step()
    scheduler.step()
    optimizer.zero_grad()

    # Log
    pbar.update()
    pbar.set_postfix(
      loss=f"{batch_loss:.2f}",
      accuracy=f"{batch_accuracy:.2f}",
      step=step + 1,
    )

    # Do validation
    if (step + 1) % valid_steps == 0:
      pbar.close()

      valid_accuracy = valid(valid_loader, model, criterion, device)

      # keep the best model
      if valid_accuracy > best_accuracy:
        best_accuracy = valid_accuracy
        best_state_dict = model.state_dict()

      pbar = tqdm(total=valid_steps, ncols=0, desc="Train", unit=" step")

    # Save the best model so far.
    if (step + 1) % save_steps == 0 and best_state_dict is not None:
      torch.save(best_state_dict, save_path)
      pbar.write(f"Step {step + 1}, best model saved. (accuracy={best_accuracy:.4f})")

  pbar.close()


if __name__ == "__main__":
  main(**parse_args())


[Info]: Use cuda now!
[Info]: Finish loading data!
[Info]: Finish creating model!


Train: 100% 2000/2000 [02:04<00:00, 16.10 step/s, accuracy=0.31, loss=3.77, step=2000]
Valid: 100% 6944/6944 [00:09<00:00, 696.28 uttr/s, accuracy=0.19, loss=4.06]
Train: 100% 2000/2000 [02:02<00:00, 16.37 step/s, accuracy=0.44, loss=2.32, step=4000]
Valid: 100% 6944/6944 [00:09<00:00, 767.83 uttr/s, accuracy=0.44, loss=2.46] 
Train: 100% 2000/2000 [02:02<00:00, 16.30 step/s, accuracy=0.66, loss=1.85, step=6000]
Valid: 100% 6944/6944 [00:10<00:00, 650.73 uttr/s, accuracy=0.56, loss=1.87]
Train: 100% 2000/2000 [02:02<00:00, 16.39 step/s, accuracy=0.72, loss=1.34, step=8000]
Valid: 100% 6944/6944 [00:08<00:00, 827.44 uttr/s, accuracy=0.62, loss=1.57] 
Train: 100% 2000/2000 [02:02<00:00, 16.30 step/s, accuracy=0.59, loss=1.36, step=1e+4]
Valid: 100% 6944/6944 [00:11<00:00, 629.91 uttr/s, accuracy=0.66, loss=1.40]
Train:   0% 3/2000 [00:00<02:51, 11.62 step/s, accuracy=0.78, loss=0.79, step=1e+4]

Step 10000, best model saved. (accuracy=0.6635)


Train: 100% 2000/2000 [02:06<00:00, 15.77 step/s, accuracy=0.78, loss=0.87, step=12000]
Valid: 100% 6944/6944 [00:09<00:00, 756.12 uttr/s, accuracy=0.70, loss=1.23]
Train: 100% 2000/2000 [02:05<00:00, 15.97 step/s, accuracy=0.91, loss=0.44, step=14000]
Valid: 100% 6944/6944 [00:10<00:00, 650.85 uttr/s, accuracy=0.74, loss=1.07]
Train: 100% 2000/2000 [02:05<00:00, 15.96 step/s, accuracy=0.69, loss=0.85, step=16000]
Valid: 100% 6944/6944 [00:08<00:00, 798.53 uttr/s, accuracy=0.75, loss=1.03] 
Train: 100% 2000/2000 [02:04<00:00, 16.10 step/s, accuracy=0.78, loss=0.74, step=18000]
Valid: 100% 6944/6944 [00:11<00:00, 610.33 uttr/s, accuracy=0.78, loss=0.92]
Train: 100% 2000/2000 [02:01<00:00, 16.40 step/s, accuracy=0.72, loss=1.06, step=2e+4]
Valid: 100% 6944/6944 [00:09<00:00, 723.58 uttr/s, accuracy=0.78, loss=0.90]
Train:   0% 3/2000 [00:00<02:40, 12.42 step/s, accuracy=0.81, loss=0.71, step=2e+4]

Step 20000, best model saved. (accuracy=0.7797)


...

Step 50000, best model saved. (accuracy=0.9003)


Train: 100% 2000/2000 [02:05<00:00, 15.88 step/s, accuracy=0.94, loss=0.23, step=52000]
Valid: 100% 6944/6944 [00:09<00:00, 713.05 uttr/s, accuracy=0.90, loss=0.41] 
Train: 100% 2000/2000 [02:05<00:00, 15.92 step/s, accuracy=0.94, loss=0.21, step=54000]
Valid: 100% 6944/6944 [00:10<00:00, 631.76 uttr/s, accuracy=0.91, loss=0.40]
Train: 100% 2000/2000 [02:03<00:00, 16.14 step/s, accuracy=0.91, loss=0.18, step=56000]
Valid: 100% 6944/6944 [00:09<00:00, 743.36 uttr/s, accuracy=0.91, loss=0.38]
Train: 100% 2000/2000 [02:02<00:00, 16.29 step/s, accuracy=0.97, loss=0.15, step=58000]
Valid: 100% 6944/6944 [00:10<00:00, 661.32 uttr/s, accuracy=0.91, loss=0.37]
Train: 100% 2000/2000 [02:03<00:00, 16.24 step/s, accuracy=0.94, loss=0.22, step=6e+4]
Valid: 100% 6944/6944 [00:08<00:00, 775.59 uttr/s, accuracy=0.92, loss=0.34]
Train:   0% 3/2000 [00:00<02:21, 14.09 step/s, accuracy=0.91, loss=0.29, step=6e+4]

Step 60000, best model saved. (accuracy=0.9209)


Train: 100% 2000/2000 [02:02<00:00, 16.29 step/s, accuracy=0.94, loss=0.19, step=62000]
Valid: 100% 6944/6944 [00:10<00:00, 645.39 uttr/s, accuracy=0.92, loss=0.36]
Train: 100% 2000/2000 [02:02<00:00, 16.28 step/s, accuracy=0.94, loss=0.21, step=64000]
Valid: 100% 6944/6944 [00:08<00:00, 798.43 uttr/s, accuracy=0.92, loss=0.36] 
Train: 100% 2000/2000 [02:03<00:00, 16.23 step/s, accuracy=1.00, loss=0.04, step=66000]
Valid: 100% 6944/6944 [00:10<00:00, 643.71 uttr/s, accuracy=0.91, loss=0.36]
Train: 100% 2000/2000 [02:03<00:00, 16.24 step/s, accuracy=0.97, loss=0.12, step=68000]
Valid: 100% 6944/6944 [00:09<00:00, 764.67 uttr/s, accuracy=0.91, loss=0.37] 
Train: 100% 2000/2000 [02:02<00:00, 16.29 step/s, accuracy=0.97, loss=0.13, step=7e+4]
Valid: 100% 6944/6944 [00:10<00:00, 668.30 uttr/s, accuracy=0.91, loss=0.36]
Train:   0% 0/2000 [00:00<?, ? step/s]


Step 70000, best model saved. (accuracy=0.9209)


# Inference

## Dataset of inference

In [26]:
import os
import json
import torch
from pathlib import Path
from torch.utils.data import Dataset


class InferenceDataset(Dataset):
  def __init__(self, data_dir):
    testdata_path = Path(data_dir) / "testdata.json"
    metadata = json.load(testdata_path.open())
    self.data_dir = data_dir
    self.data = metadata["utterances"]

  def __len__(self):
    return len(self.data)

  def __getitem__(self, index):
    utterance = self.data[index]
    feat_path = utterance["feature_path"]
    mel = torch.load(os.path.join(self.data_dir, feat_path))

    return feat_path, mel


def inference_collate_batch(batch):
  """Collate a batch of data."""
  feat_paths, mels = zip(*batch)

  return feat_paths, torch.stack(mels)


## Main funcrion of Inference

In [27]:
import json
import csv
from pathlib import Path
from tqdm.notebook import tqdm

import torch
from torch.utils.data import DataLoader

def parse_args():
  """arguments"""
  config = {
    "data_dir": "/content/dataset/Dataset",
    "model_path": "./model.ckpt",
    "output_path": "./output.csv",
  }

  return config


def main(
  data_dir,
  model_path,
  output_path,
):
  """Main function."""
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
  print(f"[Info]: Use {device} now!")

  mapping_path = Path(data_dir) / "mapping.json"
  mapping = json.load(mapping_path.open())

  dataset = InferenceDataset(data_dir)
  dataloader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=False,
    drop_last=False,
    num_workers=8,
    collate_fn=inference_collate_batch,
  )
  print(f"[Info]: Finish loading data!",flush = True)

  speaker_num = len(mapping["id2speaker"])
  model = Classifier(n_spks=speaker_num).to(device)
  model.load_state_dict(torch.load(model_path))
  model.eval()
  print(f"[Info]: Finish creating model!",flush = True)

  results = [["Id", "Category"]]
  for feat_paths, mels in tqdm(dataloader):
    with torch.no_grad():
      mels = mels.to(device)
      outs = model(mels)
      preds = outs.argmax(1).cpu().numpy()
      for feat_path, pred in zip(feat_paths, preds):
        results.append([feat_path, mapping["id2speaker"][str(pred)]])

  with open(output_path, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerows(results)


if __name__ == "__main__":
  main(**parse_args())


[Info]: Use cuda now!
[Info]: Finish loading data!
[Info]: Finish creating model!


  0%|          | 0/6000 [00:00<?, ?it/s]